# <font color="#418FDE" size="6.5" uppercase>**Embedded Sequential**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Apply sequential forward and backward feature selection. 
- Use model-based feature selection with linear and tree estimators. 
- Integrate feature selection into pipelines while considering stability and correlation. 


## **1. Sequential Feature Selection**

### **1.1. Forward Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_01_01.jpg?v=1784017095" width="250">



>* Start empty, then add strongest features
>* Stop when targets or gains are reached

>* Build smaller, interpretable feature sets
>* Greedy choices depend on earlier selections

>* Validate selection on separate data
>* Interpret correlated features with caution



In [ ]:
#@title Python Code - Forward Selection

# Demonstrate forward selection with cross-validation.
# Add features one at a time greedily.
# Show selected features and test accuracy.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

# Check that features and labels align.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split before selection to protect the test set.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)

# Scale features, then greedily select five predictors.
base_model = LogisticRegression(max_iter=1000, random_state=42)
selector = SequentialFeatureSelector(
    base_model,
    n_features_to_select=5,
    direction="forward",
    scoring="accuracy",
    cv=5,
)

# Put selection inside the pipeline to avoid leakage.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("select", selector), ("model", base_model)]
)
pipeline.fit(X_train, y_train)

# Read the selected feature mask after fitting.
selected_mask = pipeline.named_steps["select"].get_support()
selected_features = feature_names[selected_mask]

# Evaluate only once on untouched test data.
y_pred = pipeline.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("scikit-learn version:", sklearn.__version__)
print("Forward selection chose 5 of", X.shape[1], "features.")
print("Selected features:", ", ".join(selected_features))
print("Test accuracy:", round(test_accuracy, 3))



### **1.2. Backward Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_01_02.jpg?v=1784017097" width="250">



>* Start with all features, remove weakest ones
>* Stop when simpler model stays strong

>* Evaluates features within the full model
>* Removes weak or redundant variables

>* Can be slow and sample-sensitive
>* Use validation; interpret correlated features cautiously



In [ ]:
#@title Python Code - Backward Selection

# Demonstrate backward sequential feature selection.
# Remove weak features using cross-validation.
# Compare accuracy before and after selection.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

# Check that features and labels match.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data before fitting any preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Use one simple model for both comparisons.
base_model = LogisticRegression(max_iter=5000, random_state=42)
full_pipeline = Pipeline(
    [("scale", StandardScaler()), ("model", base_model)]
)

# Fit the model with all available features.
full_pipeline.fit(X_train, y_train)
full_accuracy = accuracy_score(y_test, full_pipeline.predict(X_test))

# Backward selection starts full, then removes features.
selector = SequentialFeatureSelector(
    base_model, n_features_to_select=8, direction="backward", cv=5
)
selection_pipeline = Pipeline(
    [("scale", StandardScaler()), ("select", selector), ("model", base_model)]
)

# Fit selection only on training data.
selection_pipeline.fit(X_train, y_train)
selected_accuracy = accuracy_score(y_test, selection_pipeline.predict(X_test))

# Read the selected feature names from the fitted selector.
selected_mask = selection_pipeline.named_steps["select"].get_support()
selected_features = feature_names[selected_mask]

print("scikit-learn version:", sklearn.__version__)
print("All features used:", X.shape[1])
print("Selected features used:", len(selected_features))
print("Test accuracy with all features:", round(full_accuracy, 3))
print("Test accuracy after backward selection:", round(selected_accuracy, 3))
print("Selected features:", ", ".join(selected_features[:8]))



### **1.3. Feature Count Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_01_03.jpg?v=1784017099" width="250">



>* Set a clear stopping feature count
>* Balance accuracy, simplicity, cost, and interpretability

>* Compare subset sizes for meaningful gains
>* Avoid unstable gains from greedy choices

>* Consider feature costs and availability
>* Balance evidence with practical model constraints



In [ ]:
#@title Python Code - Feature Count Limits

# Compare feature count limits in sequential selection.
# Forward selection adds predictors one by one.
# The plot shows when accuracy stops improving.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
feature_names = data.feature_names

# Keep the example small and fast for beginners.
selected_columns = np.arange(10)
X_small = X[:, selected_columns]
y = data.target

# Validate the feature matrix before modeling.
if X_small.shape[1] != 10:
    raise ValueError("Expected exactly ten candidate features.")

print("scikit-learn version:", sklearn.__version__)
print("Candidate features:", X_small.shape[1])

feature_counts = [1, 2, 3, 4, 5, 6]
mean_scores = []
selected_feature_lists = []

# Test several stopping limits for forward selection.
for count in feature_counts:
    base_model = LogisticRegression(max_iter=1000, random_state=42)
    selector = SequentialFeatureSelector(
        base_model,
        n_features_to_select=count,
        direction="forward",
        cv=3,
    )

    pipeline = make_pipeline(StandardScaler(), selector, base_model)
    scores = cross_val_score(pipeline, X_small, y, cv=3, scoring="accuracy")
    mean_scores.append(scores.mean())

    fitted_pipeline = pipeline.fit(X_small, y)
    mask = fitted_pipeline.named_steps["sequentialfeatureselector"].get_support()
    selected_feature_lists.append(feature_names[selected_columns][mask])

best_index = int(np.argmax(mean_scores))
best_count = feature_counts[best_index]
best_score = mean_scores[best_index]

print("Best tested limit:", best_count, "features")
print("Best mean CV accuracy:", round(best_score, 3))
print("Selected at best limit:", list(selected_feature_lists[best_index]))

# Plot accuracy against the allowed number of features.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(feature_counts, mean_scores, marker="o")
ax.set_title("Forward Selection: Feature Count Limit")
ax.set_xlabel("Allowed number of selected features")
ax.set_ylabel("Mean cross-validation accuracy")
ax.set_xticks(feature_counts)
ax.grid(True, alpha=0.3)

plt.show()



## **2. Embedded Feature Selection**

### **2.1. Model Based Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_02_01.jpg?v=1784017093" width="250">



>* Model learns which features matter most
>* Selected features simplify the final model

>* Linear coefficients need scaling before interpretation
>* Tree importances show useful splitting features

>* Select features using learned model importance
>* Validate selection within the full pipeline



In [ ]:
#@title Python Code - Model Based Selection

# Demonstrate embedded model based feature selection.
# Compare linear and tree importance signals.
# Show selected features and test accuracy.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = np.array(data.feature_names)

# Validate the basic feature and target shapes.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target length.")

# Split before selection to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Linear selection uses scaled coefficients as importance.
linear_pipeline = Pipeline(
    [
        ("scale", StandardScaler()),
        ("select", SelectFromModel(LogisticRegression(max_iter=5000))),
        ("model", LogisticRegression(max_iter=5000)),
    ]
)
linear_pipeline.fit(X_train, y_train)

# Tree selection uses impurity based feature importances.
tree_pipeline = Pipeline(
    [
        ("select", SelectFromModel(RandomForestClassifier(random_state=42))),
        ("model", RandomForestClassifier(random_state=42)),
    ]
)
tree_pipeline.fit(X_train, y_train)

# Read the selected feature masks from each pipeline.
linear_mask = linear_pipeline.named_steps["select"].get_support()
tree_mask = tree_pipeline.named_steps["select"].get_support()

linear_features = feature_names[linear_mask][:5]
tree_features = feature_names[tree_mask][:5]

linear_accuracy = accuracy_score(y_test, linear_pipeline.predict(X_test))
tree_accuracy = accuracy_score(y_test, tree_pipeline.predict(X_test))

print("scikit-learn version:", sklearn.__version__)
print("Linear selected feature count:", int(linear_mask.sum()))
print("Linear first selected features:", ", ".join(linear_features))
print("Linear test accuracy:", round(linear_accuracy, 3))
print("Tree selected feature count:", int(tree_mask.sum()))
print("Tree first selected features:", ", ".join(tree_features))
print("Tree test accuracy:", round(tree_accuracy, 3))



### **2.2. Lasso Feature Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_02_02.jpg?v=1784017091" width="250">



>* Lasso selects features while fitting linear models
>* Zeroed coefficients create simpler interpretable models

>* Tune regularization to balance feature selection
>* Standardize features before applying Lasso

>* Correlated features can make Lasso choices unstable
>* Validate selections with splits and domain knowledge



In [ ]:
#@title Python Code - Lasso Feature Selection

# This example demonstrates Lasso feature selection.
# Lasso can shrink weak coefficients to zero.
# Selected features are shown after proper scaling.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a small regression dataset with only three useful features.
features, target, true_coefficients = make_regression(
    n_samples=180,
    n_features=8,
    n_informative=3,
    coef=True,
    noise=12.0,
    random_state=42,
)

# Give the columns beginner-friendly names.
feature_names = np.array([
    "age",
    "income",
    "visits",
    "score",
    "calls",
    "distance",
    "tenure",
    "discount",
])

# Check that the generated data has the expected shape.
if features.shape != (180, 8):
    raise ValueError("The feature matrix shape is not as expected.")

# Split data before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# StandardScaler makes Lasso treat feature scales fairly.
lasso_selector = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectFromModel(Lasso(alpha=0.12, max_iter=5000))),
])

# Fit the embedded selector on training data only.
lasso_selector.fit(X_train, y_train)
selected_mask = lasso_selector.named_steps["selector"].get_support()
selected_features = feature_names[selected_mask]

# Collect coefficients so zero values are easy to see.
coefficients = lasso_selector.named_steps["selector"].estimator_.coef_
summary = pd.DataFrame({
    "feature": feature_names,
    "coef": np.round(coefficients, 2),
    "selected": selected_mask,
})

print("scikit-learn version:", sklearn.__version__)
print("Selected feature count:", int(selected_mask.sum()))
print("Selected features:", ", ".join(selected_features))
print(summary.sort_values("coef", key=np.abs, ascending=False).head(5).to_string(index=False))



### **2.3. Tree Based Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_02_03.jpg?v=1784017089" width="250">



>* Trees split data using predictive features
>* Ensembles rank features during model fitting

>* Trees capture nonlinear patterns and interactions
>* They reveal subgroup-specific predictive signals

>* Importance scores can be biased or unstable
>* Validate selections with repeats and domain knowledge



In [ ]:
#@title Python Code - Tree Based Selection

# Demonstrate tree based embedded feature selection.
# Feature importances come from fitted trees.
# Selected features should preserve useful accuracy.

import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = np.array(data.feature_names)

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so selection is learned only from training rows.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Fit one tree ensemble that learns feature importances internally.
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

# Keep features whose importance is at least the median importance.
selector = SelectFromModel(forest, threshold="median", prefit=True)
X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

# Train the same model type using only selected features.
selected_forest = RandomForestClassifier(n_estimators=100, random_state=42)
selected_forest.fit(X_train_selected, y_train)

# Compare accuracy before and after embedded selection.
full_accuracy = accuracy_score(y_test, forest.predict(X_test))
selected_accuracy = accuracy_score(y_test, selected_forest.predict(X_test_selected))

# Identify the five most important original features.
importance_order = np.argsort(forest.feature_importances_)[::-1]
top_indices = importance_order[:5]
top_names = feature_names[top_indices]
top_scores = forest.feature_importances_[top_indices]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Original features: {X.shape[1]}")
print(f"Selected features: {X_train_selected.shape[1]}")
print(f"Accuracy with all features: {full_accuracy:.3f}")
print(f"Accuracy with selected features: {selected_accuracy:.3f}")

# Plot the strongest tree based importance scores.
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(top_names[::-1], top_scores[::-1], color="forestgreen")
ax.set_title("Top Random Forest Feature Importances")
ax.set_xlabel("Importance score")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()



## **3. Selection Workflows**

### **3.1. Threshold Based Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_03_01.jpg?v=1784017083" width="250">



>* Keep features above model importance cutoffs
>* Simplify models and reduce noisy predictors

>* Balance threshold strictness with useful information
>* Choose thresholds inside repeatable training pipelines

>* Correlated features can distort importance scores
>* Check selection stability with validation and expertise



In [ ]:
#@title Python Code - Threshold Based Selection

# This script demonstrates threshold based feature selection.
# A pipeline prevents leakage during model evaluation.
# Selected features reveal stability and correlation effects.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a small dataset with correlated duplicate features.
features, target = make_classification(
    n_samples=600,
    n_features=12,
    n_informative=4,
    n_redundant=4,
    random_state=42,
)

# Name features so selected columns are easy to read.
feature_names = np.array(["feature_" + str(i) for i in range(features.shape[1])])

# Check the dataset shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split first so selection learns only from training data.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# Use one linear model for both selection and prediction.
base_model = LogisticRegression(max_iter=1000, random_state=42)

# Build a leakage-safe workflow with scaling and selection.
pipeline = Pipeline(
    [("scale", StandardScaler()),
     ("select", SelectFromModel(base_model, threshold="median")),
     ("model", LogisticRegression(max_iter=1000, random_state=42))]
)

# Fit the whole workflow only on the training split.
pipeline.fit(X_train, y_train)

# Read the selected feature mask from the fitted selector.
selected_mask = pipeline.named_steps["select"].get_support()
selected_features = feature_names[selected_mask]

# Evaluate the final model on untouched test data.
predicted = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predicted)

print("scikit-learn version:", sklearn.__version__)
print("Threshold rule: keep coefficients at or above the median importance.")
print("Selected feature count:", int(selected_mask.sum()), "of", features.shape[1])
print("Selected features:", ", ".join(selected_features[:10]))
print("Test accuracy:", round(accuracy, 3))



### **3.2. Pipeline Feature Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_03_02.jpg?v=1784017085" width="250">



>* Make selection part of the pipeline
>* Select features within each training fold

>* Keep preprocessing and selection consistently ordered
>* Compare complete workflows within validation

>* Track feature stability across validation folds
>* Interpret correlated selections with domain caution



In [ ]:
#@title Python Code - Pipeline Feature Selection

# This example keeps selection inside cross-validation.
# Correlated features can make selections vary.
# The output compares accuracy and stability.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# A fixed seed makes the generated data reproducible.
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=3,
    n_redundant=3,
    random_state=42,
)

# Feature names make the selected columns easier to read.
feature_names = np.array(["feature_" + str(i) for i in range(X.shape[1])])
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target values.")

# The selector is learned inside each training fold.
base_model = LogisticRegression(max_iter=1000, random_state=42)
selector = SequentialFeatureSelector(
    base_model,
    n_features_to_select=4,
    direction="forward",
    cv=3,
)

# The pipeline prevents scaling and selection leakage.
pipeline = Pipeline(
    [("scale", StandardScaler()), ("select", selector), ("model", base_model)]
)

# Cross-validation evaluates the whole workflow repeatedly.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []
selection_counts = np.zeros(X.shape[1], dtype=int)

for train_index, test_index in folds.split(X, y):
    pipeline.fit(X[train_index], y[train_index])
    accuracies.append(pipeline.score(X[test_index], y[test_index]))
    selected = pipeline.named_steps["select"].get_support()
    selection_counts = selection_counts + selected.astype(int)

# Frequent selection suggests more stable predictors.
order = np.argsort(-selection_counts)
top_features = feature_names[order[:5]]
top_counts = selection_counts[order[:5]]

print("scikit-learn version:", sklearn.__version__)
print("Mean pipeline accuracy:", round(float(np.mean(accuracies)), 3))
print("Selected in folds:", dict(zip(top_features.tolist(), top_counts.tolist())))



### **3.3. Selection Stability**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_B/image_03_03.jpg?v=1784017087" width="250">



>* Check whether selected features stay consistent
>* Interpret unstable selections with caution

>* Repeat selection within each training split
>* Check whether chosen features remain reliable

>* Correlated features can make selection unstable
>* Use diagnostics, grouping, and domain knowledge



In [ ]:
#@title Python Code - Selection Stability

# This script demonstrates feature selection stability.
# Correlated predictors can swap across folds.
# Selection frequencies reveal more than one split.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Create a small classification dataset with redundant predictors.
features, target = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=3,
    n_redundant=3,
    random_state=42,
)

# Name features so the stability table is easy to read.
feature_names = np.array([f"feature_{number}" for number in range(8)])
if features.shape[1] != len(feature_names):
    raise ValueError("Feature names must match the data columns.")

# Repeat selection inside each training fold only.
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
selection_counts = np.zeros(len(feature_names), dtype=int)

for train_index, test_index in folds.split(features, target):
    train_features = features[train_index]
    train_target = target[train_index]

    model = LogisticRegression(max_iter=1000, random_state=42)
    selector = SequentialFeatureSelector(
        model,
        n_features_to_select=3,
        direction="forward",
        cv=3,
    )

    workflow = Pipeline([
        ("scale", StandardScaler()),
        ("select", selector),
        ("model", model),
    ])
    workflow.fit(train_features, train_target)

    chosen = workflow.named_steps["select"].get_support()
    selection_counts = selection_counts + chosen.astype(int)

# Convert counts into selection frequencies across folds.
stability = pd.DataFrame({
    "feature": feature_names,
    "selected_folds": selection_counts,
    "frequency": selection_counts / folds.get_n_splits(),
})
stability = stability.sort_values("frequency", ascending=False)

print(f"scikit-learn version: {sklearn.__version__}")
print("Selection frequency across five training folds:")
print(stability.head(5).to_string(index=False))

# Plot frequencies to show stable and unstable selections.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(stability["feature"], stability["frequency"], color="steelblue")
ax.set_title("Feature Selection Stability Across Folds")
ax.set_xlabel("Feature")
ax.set_ylabel("Selection frequency")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Embedded Sequential**</font>


In this lecture, you learned to:
- Apply sequential forward and backward feature selection. 
- Use model-based feature selection with linear and tree estimators. 
- Integrate feature selection into pipelines while considering stability and correlation. 

In the next Module (Module 21), we will go over 'Neural Networks'